In [1]:


import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [2]:


text = """
Machine learning is a powerful field of artificial intelligence that allows
computers to learn patterns from data and make predictions without being
explicitly programmed for every possible situation. In recent years machine
learning has become an important part of many applications including search
engines recommendation systems medical diagnosis fraud detection autonomous
vehicles and natural language processing. A machine learning model learns
from examples and attempts to discover useful relationships between the input
data and the desired output. The quality of the data is extremely important
because a model can only learn useful patterns when the training data contains
meaningful information.

Deep learning is a subfield of machine learning that uses artificial neural
networks with multiple layers. These networks are capable of learning complex
patterns from large amounts of data. Neural networks are inspired by the way
biological neurons process information, although modern artificial neural
networks are mathematical models rather than exact representations of the
human brain. A neural network usually contains an input layer hidden layers
and an output layer. During training the network adjusts its parameters so
that its predictions become closer to the expected outputs.

One important type of neural network is the recurrent neural network. Recurrent
neural networks are designed to process sequential information. Unlike a
traditional neural network, an RNN maintains information from previous time
steps and uses that information while processing the current input. This makes
RNNs useful for tasks involving sequences such as language modeling speech
recognition time series prediction and text classification.

However, traditional recurrent neural networks can have difficulty learning
long term dependencies. When a sequence becomes long, information from earlier
parts of the sequence can gradually disappear during training. This problem is
related to the vanishing gradient problem. Another problem is the exploding
gradient problem, where gradients become extremely large and make training
unstable.

Long short term memory networks, commonly called LSTMs, were introduced to
address some of these problems. An LSTM contains a memory cell and several
gates that control the flow of information. The forget gate decides which
information should be removed from the previous cell state. The input gate
decides which new information should be stored. The output gate controls which
information from the cell state should be passed to the hidden state.

The ability of an LSTM to maintain useful information over many time steps
makes it suitable for sequence prediction tasks. One simple application of
an LSTM is next word prediction. In next word prediction, the model receives
a sequence of words and predicts the word that is most likely to come next.

For example, suppose the training sentence is the following: machine learning
is very useful. The model can create several supervised training examples.
The input machine can have learning as its target. The input machine learning
can have is as its target. The input machine learning is can have very as its
target. Finally the input machine learning is very can have useful as its
target. In this way one sentence can produce many training examples.

The words are first converted into numerical tokens because a neural network
cannot directly process ordinary words. A tokenizer assigns a unique integer
to every word in the vocabulary. The sequences are then padded so that every
training example has the same length. The input sequences are given to an
embedding layer which converts each word index into a dense numerical vector.

The embedding vectors are then processed by the LSTM layer. The LSTM reads
the sequence one word at a time and maintains information about previous words.
At the end of the sequence the hidden representation is passed to a dense
output layer. The dense layer contains one output for every word in the
vocabulary. A softmax activation converts these outputs into probabilities.

During supervised training the actual next word is used as the target label.
The model compares its predicted probability distribution with the actual
target word and calculates a loss. Backpropagation is then used to update the
weights of the embedding layer, LSTM layer, and output layer. This process is
repeated for many batches and epochs.

After training, a user can provide a partial sentence such as machine learning
is and the model can predict the most probable next word. The predicted word
can then be added to the sentence and the process can be repeated. However,
the main purpose of this project is not to create a large text generation
system. The goal is to understand supervised next word classification using
an LSTM.

Natural language processing contains many other important tasks. Sentiment
analysis attempts to determine the emotional meaning of text. Text
classification assigns documents to predefined categories. Named entity
recognition identifies people organizations locations and other entities.
Machine translation converts text from one language into another. Question
answering systems attempt to find appropriate answers based on a question and
some context.

Understanding sequence models is useful because language naturally contains
relationships between words. The meaning of a word can depend on the words
that appear before it. For example, the word bank can have different meanings
depending on whether the sentence discusses money or a river. A sequence
model attempts to use the surrounding context to make better predictions.

The training process for a neural network is iterative. Initially the model
has randomly initialized weights and its predictions are mostly incorrect.
After each training step the model receives feedback through the loss function.
The optimizer uses the gradients of the loss to modify the model parameters.
As training continues the model gradually learns patterns that occur frequently
in the training data.

A good dataset should contain enough examples for the model to learn useful
patterns. If the dataset is too small, the model may memorize the training
sentences instead of learning general patterns. This is called overfitting.
Validation data can be used to monitor how well the model performs on examples
that were not used for updating the model weights.

For a small educational project, an LSTM with an embedding layer followed by
one LSTM layer and a dense softmax layer is enough to demonstrate the complete
next word prediction pipeline. The important concepts are tokenization,
sequence creation, padding, supervised labels, embeddings, recurrent memory,
LSTM gates, softmax classification, categorical cross entropy, backpropagation,
and prediction.

The same basic idea can later be extended to larger datasets and more
complicated architectures. More training data can improve the vocabulary and
the variety of patterns learned by the model. Multiple LSTM layers can be
used to create a deeper network. Bidirectional LSTMs can process information
from both directions when the task allows it. Attention mechanisms and
transformer architectures can also be used for more advanced natural language
processing applications.

The important point is that next word prediction can be formulated as a
supervised learning problem. Every training example contains an input sequence
and a known target word. The LSTM learns a mapping from the previous words to
the probability distribution of the next word. Once this mapping has been
learned, the model can predict the next word for a previously unseen input
sequence.
"""

print("Number of characters:", len(text))

Number of characters: 7712


In [3]:


text = text.lower()

import re

text = re.sub(r'[^a-z\s]', '', text)
text = re.sub(r'\s+', ' ', text).strip()

print(text[:1000])

machine learning is a powerful field of artificial intelligence that allows computers to learn patterns from data and make predictions without being explicitly programmed for every possible situation in recent years machine learning has become an important part of many applications including search engines recommendation systems medical diagnosis fraud detection autonomous vehicles and natural language processing a machine learning model learns from examples and attempts to discover useful relationships between the input data and the desired output the quality of the data is extremely important because a model can only learn useful patterns when the training data contains meaningful information deep learning is a subfield of machine learning that uses artificial neural networks with multiple layers these networks are capable of learning complex patterns from large amounts of data neural networks are inspired by the way biological neurons process information although modern artificial n

In [4]:

tokenizer = Tokenizer(
    num_words=None,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

print("Vocabulary size:", total_words)

print("\nFirst 20 words:")
print(list(tokenizer.word_index.items())[:20])

Vocabulary size: 407

First 20 words:
[('<OOV>', 1), ('the', 2), ('a', 3), ('to', 4), ('is', 5), ('and', 6), ('of', 7), ('can', 8), ('word', 9), ('model', 10), ('training', 11), ('learning', 12), ('for', 13), ('information', 14), ('an', 15), ('layer', 16), ('lstm', 17), ('machine', 18), ('that', 19), ('input', 20)]


In [5]:


tokens = tokenizer.texts_to_sequences([text])[0]

print("First 30 tokens:")
print(tokens[:30])

First 30 tokens:
[18, 12, 5, 3, 161, 162, 7, 70, 163, 19, 102, 164, 4, 71, 32, 23, 28, 6, 72, 54, 165, 166, 167, 168, 13, 47, 169, 170, 41, 171]


In [6]:
# ============================================
# 6. CREATE SUPERVISED TRAINING DATA
# ============================================
#
# Example:
#
# machine
# machine learning
# machine learning is
# machine learning is a
#
# Each sequence predicts its NEXT word.
#
# input                  target
# machine                learning
# machine learning       is
# machine learning is    a
#
# ============================================

input_sequences = []

for i in range(1, len(tokens)):
    sequence = tokens[:i+1]
    input_sequences.append(sequence)

print(input_sequences)

[[18, 12], [18, 12, 5], [18, 12, 5, 3], [18, 12, 5, 3, 161], [18, 12, 5, 3, 161, 162], [18, 12, 5, 3, 161, 162, 7], [18, 12, 5, 3, 161, 162, 7, 70], [18, 12, 5, 3, 161, 162, 7, 70, 163], [18, 12, 5, 3, 161, 162, 7, 70, 163, 19], [18, 12, 5, 3, 161, 162, 7, 70, 163, 19, 102], [18, 12, 5, 3, 161, 162, 7, 70, 163, 19, 102, 164], [18, 12, 5, 3, 161, 162, 7, 70, 163, 19, 102, 164, 4], [18, 12, 5, 3, 161, 162, 7, 70, 163, 19, 102, 164, 4, 71], [18, 12, 5, 3, 161, 162, 7, 70, 163, 19, 102, 164, 4, 71, 32], [18, 12, 5, 3, 161, 162, 7, 70, 163, 19, 102, 164, 4, 71, 32, 23], [18, 12, 5, 3, 161, 162, 7, 70, 163, 19, 102, 164, 4, 71, 32, 23, 28], [18, 12, 5, 3, 161, 162, 7, 70, 163, 19, 102, 164, 4, 71, 32, 23, 28, 6], [18, 12, 5, 3, 161, 162, 7, 70, 163, 19, 102, 164, 4, 71, 32, 23, 28, 6, 72], [18, 12, 5, 3, 161, 162, 7, 70, 163, 19, 102, 164, 4, 71, 32, 23, 28, 6, 72, 54], [18, 12, 5, 3, 161, 162, 7, 70, 163, 19, 102, 164, 4, 71, 32, 23, 28, 6, 72, 54, 165], [18, 12, 5, 3, 161, 162, 7, 70, 163,

In [7]:
# ============================================
# 7. PAD SEQUENCES
# ============================================

max_sequence_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_sequence_len,
    padding='pre'
)

print("Maximum sequence length:", max_sequence_len)
print("Shape:", input_sequences.shape)

Maximum sequence length: 1176
Shape: (1175, 1176)


In [8]:
# ============================================
# 8. SPLIT INPUT X AND TARGET Y
# ============================================

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nExample:")
print("X:", X[0])
print("y:", y[0])

X shape: (1175, 1175)
y shape: (1175,)

Example:
X: [ 0  0  0 ...  0  0 18]
y: 12


In [9]:
# ============================================
# 9. DECODE ONE TRAINING EXAMPLE
# ============================================

reverse_word_index = {
    value: key
    for key, value in tokenizer.word_index.items()
}

example_x = X[10]

words = []

for token in example_x:
    if token != 0:
        words.append(reverse_word_index[token])

target_word = reverse_word_index[y[10]]

print("Input:", " ".join(words))
print("Target:", target_word)

Input: machine learning is a powerful field of artificial intelligence that allows
Target: computers


In [10]:
# ============================================
# 10. BUILD LSTM MODEL
# ============================================

model = Sequential([

    Embedding(
        input_dim=total_words,
        output_dim=64,
        input_length=max_sequence_len - 1
    ),

    LSTM(128),

    Dense(
        total_words,
        activation='softmax'
    )
])

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
# ============================================
# 11. COMPILE MODEL
# ============================================

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("Model compiled successfully.")

Model compiled successfully.


In [12]:
# ============================================
# 12. TRAIN MODEL
# ============================================

history = model.fit(
    X,
    y,
    epochs=100,
    batch_size=32,
    verbose=1
)

Epoch 1/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.0698 - loss: 5.7768
Epoch 2/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.0800 - loss: 5.3820
Epoch 3/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.0800 - loss: 5.3280
Epoch 4/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.0800 - loss: 5.3085
Epoch 5/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.0800 - loss: 5.2755
Epoch 6/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - accuracy: 0.0800 - loss: 5.2361
Epoch 7/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.0809 - loss: 5.1518
Epoch 8/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.0919 - loss: 5.0154
Epoch 9/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.1038 - loss: 4.8311
Epoch 10/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.1311 - loss: 4.6387
Epoch 11/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.1506 - loss: 4.4387
Epoch 12/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step

In [ ]:
y.shape

(863,)

In [14]:
# ============================================
# 14. TOP 5 NEXT WORDS
# ============================================

def predict_top_words(seed_text, top_k=5):

    seed_text = seed_text.lower()

    sequence = tokenizer.texts_to_sequences([seed_text])[0]

    if len(sequence) == 0:
        print("Words not found in vocabulary.")
        return

    sequence = pad_sequences(
        [sequence],
        maxlen=max_sequence_len - 1,
        padding='pre'
    )

    probabilities = model.predict(
        sequence,
        verbose=0
    )[0]

    top_indices = np.argsort(probabilities)[-top_k:][::-1]

    print("\nInput:", seed_text)
    print("\nTop predictions:")

    for index in top_indices:

        word = reverse_word_index.get(
            index,
            "unknown"
        )

        probability = probabilities[index] * 100

        print(
            f"{word:20s} {probability:.2f}%"
        )

In [15]:
# ============================================
# 13. NEXT WORD PREDICTION FUNCTION
# ============================================

def predict_next_word(seed_text):

    seed_text = seed_text.lower()

    sequence = tokenizer.texts_to_sequences([seed_text])[0]

    if len(sequence) == 0:
        print("Words not found in vocabulary.")
        return

    sequence = pad_sequences(
        [sequence],
        maxlen=max_sequence_len - 1,
        padding='pre'
    )

    probabilities = model.predict(
        sequence,
        verbose=0
    )[0]

    predicted_token = np.argmax(probabilities)

    predicted_word = reverse_word_index.get(
        predicted_token,
        "unknown"
    )

    print("Input:", seed_text)
    print("Predicted next word:", predicted_word)

In [16]:
# ============================================
# TOP 100 TEST CASES
# ============================================

test_cases = [
    "machine learning",
    "machine learning is",
    "machine learning has",
    "machine learning model",
    "machine learning becomes",
    "machine learning applications",
    "deep learning",
    "deep learning is",
    "deep learning can",
    "deep learning applications",

    "natural language",
    "natural language processing",
    "natural language contains",
    "natural language tasks",
    "natural language applications",

    "neural network",
    "neural networks",
    "neural networks are",
    "neural networks can",
    "neural network usually",
    "neural network learns",
    "neural network training",

    "the model",
    "the model learns",
    "the model can",
    "the model receives",
    "the model predicts",
    "the model parameters",

    "the training",
    "the training data",
    "the training process",
    "the training examples",
    "the training corpus",

    "artificial intelligence",
    "artificial intelligence that",
    "artificial neural",
    "artificial neural networks",

    "supervised learning",
    "supervised learning problem",
    "supervised training",
    "supervised next word",

    "next word",
    "next word prediction",
    "next word prediction is",
    "next word can",

    "recurrent neural",
    "recurrent neural networks",
    "recurrent neural networks are",
    "recurrent neural network",

    "long short",
    "long short term",
    "long short term memory",
    "long short term memory networks",

    "memory networks",
    "memory cell",
    "memory cell and",
    "memory over",

    "the forget",
    "the forget gate",
    "the input",
    "the input gate",
    "the output",
    "the output gate",

    "forget gate",
    "forget gate decides",
    "input gate",
    "input gate decides",
    "output gate",
    "output gate controls",

    "sequence model",
    "sequence models",
    "sequence prediction",
    "sequence prediction tasks",
    "sequential information",
    "sequential information and",

    "training data",
    "training data contains",
    "training data should",
    "training examples",
    "training examples are",

    "embedding layer",
    "embedding vectors",
    "embedding vectors are",
    "embedding layer which",

    "dense output",
    "dense output layer",
    "dense layer",
    "dense layer contains",

    "softmax activation",
    "softmax activation converts",
    "probability distribution",
    "probability distribution of",

    "backpropagation",
    "backpropagation is",
    "loss function",
    "loss function and",

    "gradient",
    "gradients",
    "gradients become",
    "vanishing gradient",
    "vanishing gradient problem",
    "exploding gradient",
    "exploding gradient problem",

    "more training",
    "more training data",
    "more data",
    "more diverse",
    "more diverse corpus"
]

# ============================================
# RUN ALL 100 TEST CASES
# ============================================

for i, text in enumerate(test_cases, 1):

    print("\n" + "=" * 70)
    print(f"TEST CASE {i}: {text}")

    predict_top_words(text, top_k=5)


TEST CASE 1: machine learning

Input: machine learning

Top predictions:
is                   97.13%
that                 1.54%
a                    0.44%
and                  0.13%
learning             0.12%

TEST CASE 2: machine learning is

Input: machine learning is

Top predictions:
a                    96.86%
is                   0.72%
uses                 0.56%
artificial           0.38%
that                 0.37%

TEST CASE 3: machine learning has

Input: machine learning has

Top predictions:
a                    98.99%
is                   0.42%
one                  0.19%
uses                 0.14%
artificial           0.10%

TEST CASE 4: machine learning model

Input: machine learning model

Top predictions:
is                   72.12%
a                    11.54%
that                 4.42%
artificial           2.17%
can                  1.91%

TEST CASE 5: machine learning becomes

Input: machine learning becomes

Top predictions:
a                    77.48%
is             